# Machine Learning for Anti-Money Laundering

***
**Master's thesis: empirical part**

**Notebook 1: Data and Exploratory Data Analysis**

Benchmark: IBM AMLworld, variants HI-Small and LI-Small (Altman et al., 2023)
***

| Notebook | Content | Chapter 4 |
| --- | --- | --- |
| **01 Data and EDA** (this notebook) | acquisition, integrity checks, temporal partition, EDA | 4.2–4.5 |
| 02 Features and feature selection | Sets A, B, C; causality test; screening | 4.6 |
| 03 Models | one section per family: search, fit, threshold, evaluation | 4.7–4.8 |
| 04 Results | comparison across families, sets and variants | 4.9 |

## Preliminaries

This notebook loads and checks the two transaction files, fixes the chronological partition,
and explores the data before any feature is constructed.

**Before the first run:** select a CPU runtime, then *Runtime → Run all*. No API key is
needed. The notebook first tries to download the four files with `kagglehub`, which works
without credentials for most public datasets. If that fails, it stops and prints what to
do instead: download the four files by hand from the Data tab of the Kaggle page (logged in
to Kaggle in the browser, one file at a time) and upload them, zipped or not, to the Drive
folder `MyDrive/aml-thesis/data/raw/`; then run again. Files found there are never
downloaded. Everything
persistent is kept in `MyDrive/aml-thesis/`, because the Colab machine is reset between
sessions and the split boundaries must survive those resets (Section 4.5).

`DRY_RUN = True` runs the whole notebook on small synthetic files, without Kaggle, in a
separate Drive folder. Its figures are meaningless; it only verifies that the code runs.

In [ ]:
# Parameters
VARIANTS    = ["HI-Small", "LI-Small"]   # both are loaded, checked and partitioned
EDA_VARIANT = "HI-Small"                 # the variant explored in Section 4 of this notebook
SEED        = 0                          # used only for plotting samples
DRY_RUN     = False

In [ ]:
# Import relevant Python packages
import os, sys, json, gc, platform, subprocess, datetime, hashlib, shutil, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from pandas import set_option
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

set_option("display.precision", 3)
set_option("display.width", 120)
set_option("display.max_columns", 30)
sns.set_theme(context="notebook", style="whitegrid", palette="viridis")

In [ ]:
# Google Drive, project folder and a record of the computational environment
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive")
except ImportError:                      # outside Colab: work next to the notebook
    BASE = Path.cwd()
PROJECT = BASE / ("aml-thesis-dryrun" if DRY_RUN else "aml-thesis")
for d in ["aml", "tests", "data/raw", "data/interim", "results/figures", "results/tables"]:
    (PROJECT / d).mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 2**30
except ImportError:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
header = (f"# recorded {datetime.datetime.now().isoformat(timespec='seconds')}\n"
          f"# python {platform.python_version()} | cpus {os.cpu_count()} | ram_gb {ram_gb:.1f}\n")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True).stdout
(PROJECT / "results" / "environment.txt").write_text(header + freeze)
print(f"project: {PROJECT}")
print(header)

### Project code

The cells below write the `aml` package and its tests to Drive; every later cell (and
notebooks 02–04) imports it from there. Pipeline logic lives only in these modules, so that
it is tested once and shared; the exploratory plots further down are written inline. After
editing one of these cells, run it again together with the cell that follows them.
Library versions are recorded rather than pinned, since downgrading Colab's preinstalled
packages forces a runtime restart; the code is tested on pandas 2.2 and 3.0.

In [ ]:
%%writefile aml/__init__.py
"""Experimental pipeline for the AMLworld comparative evaluation (Chapter 4)."""


In [ ]:
%%writefile aml/config.py
"""Single source of fixed experimental parameters (Sections 4.2, 4.5, 4.9, 4.10).

Nothing here is tuned. Any change after the first run is a deviation from the
specification and must be recorded in Appendix A.
"""
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
DATA_RAW = ROOT / "data" / "raw"          # Kaggle files, unmodified
DATA_INTERIM = ROOT / "data" / "interim"  # typed, sorted, split frames (Parquet)
RESULTS = ROOT / "results"                # integrity reports, split boundaries, metrics

VARIANTS = ("HI-Small", "LI-Small")

# Header as distributed: the account column name is repeated (Section 4.4).
RAW_HEADER = ["Timestamp", "From Bank", "Account", "To Bank", "Account",
              "Amount Received", "Receiving Currency", "Amount Paid",
              "Payment Currency", "Payment Format", "Is Laundering"]

# Names applied by position after the raw header has been verified (Listing 4.1).
COLUMNS = ["Timestamp", "From Bank", "src_account", "To Bank", "dst_account",
           "Amount Received", "Receiving Currency", "Amount Paid",
           "Payment Currency", "Payment Format", "Is Laundering"]

TIMESTAMP_FORMAT = "%Y/%m/%d %H:%M"

# Published composition (Table 4.1; Altman et al., 2023, Appendix Table 4).
# Counts are rounded in the source, so they are compared with a tolerance.
PUBLISHED = {
    "HI-Small": {"transactions": 5_000_000, "laundering": 3_600, "ratio_1_in": 981},
    "LI-Small": {"transactions": 7_000_000, "laundering": 4_000, "ratio_1_in": 1_942},
}
COUNT_TOLERANCE = 0.05   # rounded published counts ("5M", "3.6K")
RATIO_TOLERANCE = 0.01   # published 1-in-N ratio is given to the unit

SPLIT_QUANTILES = (0.70, 0.85)   # train | validation | test (Section 4.5)
SEEDS = (0, 1, 2, 3, 4)          # Section 4.9

# Columns that may never enter a feature matrix (Sections 4.4 and 4.6).
# Raw identifiers and the raw timestamp are inputs to feature construction, not features.
NON_FEATURE_COLS = ["Is Laundering", "typology", "attempt_id", "block", "file_row",
                    "Timestamp", "src_account", "dst_account", "From Bank", "To Bank"]

# Kaggle dataset (Section 4.2). The version is resolved on the first download and then
# pinned in results/DATA_VERSION.json.
DATASET_HANDLE = "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"


In [ ]:
%%writefile aml/acquire.py
"""Acquisition of the raw files (Section 4.2): kagglehub, or files placed by hand.

If the four files (or zip archives containing them) are already in raw_dir, nothing is
downloaded, so a manual download through the Kaggle website works without any API key.

Only the download goes through kagglehub. Parsing stays in `load.read_transactions`,
which verifies the repeated `Account` header before renaming (Section 4.4); the patterns
file is not tabular and cannot be read by kagglehub's pandas adapter in any case.
"""
import datetime
import json
import re
import shutil
import zipfile
from pathlib import Path

from . import config


def _version_in(path: Path):
    m = re.search(r"[/\\]versions[/\\](\d+)(?:[/\\]|$)", str(path))
    return int(m.group(1)) if m else None


MANUAL_HELP = """\
Automatic download failed ({error}).
Download these files by hand from the Data tab of
  https://www.kaggle.com/datasets/{handle}/data
(click each file, then its download icon; do not download the whole dataset):
  {files}
and upload them, zipped or not, to the Drive folder
  {raw_dir}
then run this cell again."""


def unpack_archives(raw_dir, files) -> list:
    """Extract any wanted file found inside a zip archive in raw_dir (for manual
    downloads, which Kaggle may deliver zipped). The archives themselves are kept."""
    raw_dir, wanted, done = Path(raw_dir), set(files), []
    for z in sorted(raw_dir.glob("*.zip")):
        with zipfile.ZipFile(z) as zf:
            for member in zf.namelist():
                name = Path(member).name
                if name in wanted and not (raw_dir / name).exists():
                    with zf.open(member) as src, open(raw_dir / name, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    done.append(name)
    return done


def fetch_raw(files, raw_dir, version_file, handle: str = config.DATASET_HANDLE,
              download=None):
    """Make the four files available in raw_dir and record where they came from.

    Files already in raw_dir (or in a zip there) are used as they are. Anything missing
    is downloaded with kagglehub; its cache on the Colab machine is wiped between
    sessions, so each file is copied to raw_dir on Drive. The version resolved by the
    first download is recorded and requested explicitly afterwards, so a new upload on
    Kaggle cannot enter the experiment unnoticed. Returns the recorded version, or None
    for files placed by hand (the checksums then carry the guarantee).
    """
    raw_dir, version_file = Path(raw_dir), Path(version_file)
    raw_dir.mkdir(parents=True, exist_ok=True)
    unpack_archives(raw_dir, files)
    record = json.loads(version_file.read_text()) if version_file.exists() else None
    version = record["version"] if record else None
    missing = [f for f in files if not (raw_dir / f).exists()]

    if not missing:
        if record is None:
            version_file.parent.mkdir(parents=True, exist_ok=True)
            version_file.write_text(json.dumps({
                "handle": handle, "version": None, "source": "placed in data/raw by hand",
                "recorded": datetime.date.today().isoformat()}, indent=2))
        return version

    try:
        if download is None:
            import kagglehub
            download = kagglehub.dataset_download
        for f in missing:
            h = handle if version is None else f"{handle}/versions/{version}"
            cached = Path(download(h, path=f))
            resolved = _version_in(cached)
            if version is None and resolved is not None:
                version = resolved
                version_file.parent.mkdir(parents=True, exist_ok=True)
                version_file.write_text(json.dumps({
                    "handle": handle, "version": version, "source": "kagglehub",
                    "recorded": datetime.date.today().isoformat()}, indent=2))
            elif None not in (version, resolved) and resolved != version:
                raise ValueError(f"{f}: received version {resolved}, pinned {version}")
            if cached.is_dir():
                cached = cached / f
            if cached.suffix == ".zip":
                with zipfile.ZipFile(cached) as zf:
                    zf.extract(f, raw_dir)
            else:
                shutil.copy2(cached, raw_dir / f)
            print(f"{f}: version {resolved}, copied to {raw_dir}")
    except ValueError:
        raise
    except Exception as e:  # authentication, network, missing package
        still = [f for f in files if not (raw_dir / f).exists()]
        raise RuntimeError(MANUAL_HELP.format(error=type(e).__name__, handle=handle,
                                              files="\n  ".join(still),
                                              raw_dir=raw_dir)) from None
    return version


In [ ]:
%%writefile aml/load.py
"""Loading, typing and integrity checks for one transaction file (Section 4.4)."""
import csv

import numpy as np
import pandas as pd

from . import config

AMOUNT_COLS = ["Amount Received", "Amount Paid"]
# Arrow-backed strings use a fraction of the memory of Python objects; RAM is the
# binding constraint at five to seven million records (Section 4.4).
STR_DTYPE = "string[pyarrow]"
CATEGORY_COLS = ["src_account", "dst_account", "From Bank", "To Bank",
                 "Payment Format", "Receiving Currency", "Payment Currency"]


def verify_header(path) -> None:
    """Refuse to rename by position unless the file has the documented column order.

    Positional renaming alone does not protect against a reordered file; checking the
    raw header first is what makes the renaming in Listing 4.1 safe.
    """
    with open(path, newline="", encoding="utf-8") as fh:
        header = [h.strip() for h in next(csv.reader(fh))]
    if header != config.RAW_HEADER:
        raise ValueError(f"Unexpected header in {path}:\n{header}\n"
                         f"expected:\n{config.RAW_HEADER}")


def normalise_bank(s: pd.Series) -> pd.Series:
    """Bank identifiers may be zero-padded differently across files ("0110" vs "110")."""
    s = s.astype(STR_DTYPE).str.strip().str.lstrip("0")
    return s.where(s != "", "0")


def read_transactions(path) -> pd.DataFrame:
    """Read identifiers as strings and amounts as float64.

    Categorical typing and float32 downcasting are deferred to `finalise_types`, so
    that the join with the patterns file (exact equality on every field) happens
    before any lossy conversion.
    """
    verify_header(path)
    dtypes = {c: STR_DTYPE for c in config.COLUMNS
              if c not in AMOUNT_COLS + ["Is Laundering"]}
    dtypes.update({c: "float64" for c in AMOUNT_COLS})
    dtypes["Is Laundering"] = "int32"  # int8 sums can overflow silently in groupby
    df = pd.read_csv(path, header=0, names=config.COLUMNS, dtype=dtypes)
    # errors="coerce" is not silent: unparsable timestamps are counted in the
    # integrity report rather than dropped.
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], format=config.TIMESTAMP_FORMAT,
                                     errors="coerce")
    df["From Bank"] = normalise_bank(df["From Bank"])
    df["To Bank"] = normalise_bank(df["To Bank"])
    df["file_row"] = np.arange(len(df), dtype=np.int32)  # audit trail to the raw file
    return df


def finalise_types(df: pd.DataFrame) -> pd.DataFrame:
    """Memory-oriented typing (Listing 4.1) and a stable chronological sort.

    The sort is stable, so transactions sharing a timestamp keep their file order
    and every rebuild yields the same row order.
    """
    for c in CATEGORY_COLS:
        df[c] = df[c].astype("category")
    for c in AMOUNT_COLS:
        df[c] = pd.to_numeric(df[c], downcast="float")
    return df.sort_values("Timestamp", kind="mergesort").reset_index(drop=True)


def _rel_dev(observed: float, published: float) -> float:
    return float(observed / published - 1.0)


def integrity_checks(df: pd.DataFrame, variant: str) -> dict:
    """The three checks of Section 4.4. Nothing is removed; every count is reported."""
    pub = config.PUBLISHED[variant]
    n = int(len(df))
    n_pos = int(df["Is Laundering"].sum())
    ratio = n / n_pos if n_pos else float("inf")
    ts = df["Timestamp"]

    report = {
        "n_records": n,
        "n_laundering": n_pos,
        "observed_ratio_1_in": round(ratio, 1),
        "published": pub,
        "dev_records": round(_rel_dev(n, pub["transactions"]), 4),
        "dev_laundering": round(_rel_dev(n_pos, pub["laundering"]), 4),
        "dev_ratio": round(_rel_dev(ratio, pub["ratio_1_in"]), 4),
        "n_unparsed_timestamps": int(ts.isna().sum()),
        "timestamp_non_decreasing": bool(ts.dropna().is_monotonic_increasing),
        "n_duplicate_records": int(df.duplicated(subset=config.COLUMNS).sum()),
        "n_nonpositive_amount_paid": int((df["Amount Paid"] <= 0).sum()),
        "first_timestamp": str(ts.min()),
        "last_timestamp": str(ts.max()),
        "span_days": round((ts.max() - ts.min()) / pd.Timedelta(days=1), 3),
    }
    report["checks"] = {
        "records_match_table_4_1": abs(report["dev_records"]) <= config.COUNT_TOLERANCE,
        "laundering_matches_table_4_1": abs(report["dev_laundering"]) <= config.COUNT_TOLERANCE,
        "ratio_matches_table_4_1": abs(report["dev_ratio"]) <= config.RATIO_TOLERANCE,
        "timestamps_parsed_and_ordered": (report["n_unparsed_timestamps"] == 0
                                          and report["timestamp_non_decreasing"]),
        "no_duplicate_records": report["n_duplicate_records"] == 0,
    }
    return report


In [ ]:
%%writefile aml/patterns.py
"""Parser for the laundering-patterns file and its join to the transactions (Section 4.4).

The typology label produced here is descriptive only (Section 5.5). It must never
enter a feature matrix; see config.NON_FEATURE_COLS.
"""
import csv

import numpy as np
import pandas as pd

from . import config
from .load import AMOUNT_COLS, STR_DTYPE, normalise_bank

TYPOLOGIES = {
    "FAN-OUT": "fan-out", "FAN-IN": "fan-in",
    "GATHER-SCATTER": "gather-scatter", "SCATTER-GATHER": "scatter-gather",
    "CYCLE": "cycle", "RANDOM": "random", "BIPARTITE": "bipartite", "STACK": "stack",
}
JOIN_COLS = config.COLUMNS[:-1]   # every recorded field except the label


def parse_patterns(path) -> pd.DataFrame:
    """Read 'BEGIN LAUNDERING ATTEMPT - <TYPE>[: detail]' ... 'END ...' blocks."""
    records, attempt, typology = [], -1, None
    with open(path, newline="", encoding="utf-8") as fh:
        for lineno, raw in enumerate(fh, start=1):
            line = raw.strip()
            if not line:
                continue
            if line.startswith("BEGIN LAUNDERING ATTEMPT"):
                attempt += 1
                label = line.split("-", 1)[1].split(":", 1)[0].strip().upper()
                typology = TYPOLOGIES.get(label, label.lower())  # unknown labels are reported
            elif line.startswith("END LAUNDERING ATTEMPT"):
                typology = None
            else:
                if typology is None:
                    raise ValueError(f"{path}:{lineno}: transaction outside an attempt block")
                fields = next(csv.reader([line]))
                if len(fields) != len(config.COLUMNS):
                    raise ValueError(f"{path}:{lineno}: expected {len(config.COLUMNS)} "
                                     f"fields, found {len(fields)}")
                records.append(fields + [attempt, typology])

    pat = pd.DataFrame(records, columns=config.COLUMNS + ["attempt_id", "typology"])
    pat["Timestamp"] = pd.to_datetime(pat["Timestamp"], format=config.TIMESTAMP_FORMAT)
    for c in AMOUNT_COLS:
        pat[c] = pat[c].astype("float64")
    pat["Is Laundering"] = pat["Is Laundering"].astype("int32")
    pat["From Bank"] = normalise_bank(pat["From Bank"])
    pat["To Bank"] = normalise_bank(pat["To Bank"])
    for c in JOIN_COLS:   # identical dtypes on both sides of the join
        if c not in AMOUNT_COLS + ["Timestamp"]:
            pat[c] = pat[c].astype(STR_DTYPE)
    return pat


def attach_typology(df: pd.DataFrame, pat: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Attach typology and attempt id to the transactions by exact equality of every field.

    Only rows sharing both a timestamp and an originating account with some pattern
    transaction can match, so the exact join runs on that small candidate subset
    rather than on all records. The result is identical to a full left join.
    """
    n = len(df)
    pmap = pat[JOIN_COLS + ["attempt_id", "typology"]]
    n_multi = int(pmap.duplicated(JOIN_COLS).sum())   # listed in more than one attempt
    pmap = pmap.drop_duplicates(JOIN_COLS, keep="first")

    cand = (df["Timestamp"].isin(pmap["Timestamp"].unique())
            & df["src_account"].isin(pmap["src_account"].unique())).to_numpy()
    sub = df.loc[cand, JOIN_COLS].copy()
    sub["_pos"] = np.flatnonzero(cand)
    m = sub.merge(pmap, on=JOIN_COLS, how="inner", validate="many_to_one")
    pos = m["_pos"].to_numpy()

    attempt = pd.Series(pd.NA, index=df.index, dtype="Int32")
    attempt.iloc[pos] = m["attempt_id"].to_numpy(dtype="int32")
    typology = np.full(n, None, dtype=object)
    typology[pos] = m["typology"].to_numpy(dtype=object)
    df["attempt_id"] = attempt
    df["typology"] = pd.Categorical(typology)

    label = df["Is Laundering"].to_numpy()
    has_typ = df["typology"].notna().to_numpy()
    report = {
        "n_pattern_rows": int(len(pat)),
        "n_attempts": int(pat["attempt_id"].nunique()),
        "attempts_by_typology": (pat.drop_duplicates("attempt_id")["typology"]
                                 .value_counts().to_dict()),
        "unknown_typology_labels": sorted(set(pat["typology"]) - set(TYPOLOGIES.values())),
        "n_pattern_transactions_in_multiple_attempts": n_multi,
        "n_pattern_transactions_unmatched": int(len(pmap)
                                                - len(m[JOIN_COLS].drop_duplicates())),
        "n_rows_with_typology": int(has_typ.sum()),
        "n_laundering_rows_without_typology": int(((label == 1) & ~has_typ).sum()),
        "n_typology_rows_labelled_licit": int((has_typ & (label == 0)).sum()),
    }
    return df, report


In [ ]:
%%writefile aml/split.py
"""Chronological partitioning, fixed once and never revised (Section 4.5, Listing 4.2)."""
import json
from pathlib import Path

import numpy as np
import pandas as pd

from . import config

BLOCKS = ["train", "val", "test"]


def compute_boundaries(ts: pd.Series, q=config.SPLIT_QUANTILES) -> dict:
    """Quantiles of the record timestamps, so the blocks hold 70/15/15 per cent of
    records, not of calendar time. interpolation="lower" makes each boundary an
    observed timestamp; with `<=`, all records sharing a minute fall in one block.
    """
    cut_train = ts.quantile(q[0], interpolation="lower")
    cut_val = ts.quantile(q[1], interpolation="lower")
    return {"quantiles": list(q), "cut_train": str(cut_train), "cut_val": str(cut_val)}


def fix_boundaries(ts: pd.Series, path: Path) -> dict:
    """Write the boundaries on the first run; on every later run, verify them."""
    bounds = compute_boundaries(ts)
    path = Path(path)
    if path.exists():
        stored = json.loads(path.read_text())
        if stored != bounds:
            raise RuntimeError(f"Split boundaries differ from those recorded in {path}: "
                               f"{stored} vs {bounds}. Section 4.5 forbids revision; "
                               "investigate before deleting the file.")
        return stored
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(bounds, indent=2))
    return bounds


def assign_block(ts: pd.Series, bounds: dict) -> pd.Series:
    cut_train = pd.Timestamp(bounds["cut_train"])
    cut_val = pd.Timestamp(bounds["cut_val"])
    block = np.select([ts <= cut_train, ts <= cut_val], ["train", "val"], default="test")
    return pd.Series(pd.Categorical(block, categories=BLOCKS, ordered=True), index=ts.index)


def block_summary(df: pd.DataFrame) -> dict:
    g = df.groupby("block", observed=True)
    out = pd.DataFrame({
        "n_records": g.size(),
        "n_laundering": g["Is Laundering"].sum().astype("int64"),
        "first": g["Timestamp"].min().astype(str),
        "last": g["Timestamp"].max().astype(str),
    })
    out["share_records"] = (out["n_records"] / out["n_records"].sum()).round(4)
    out["ratio_1_in"] = (out["n_records"] / out["n_laundering"]).round(1)
    return {b: {k: (v.item() if hasattr(v, "item") else v) for k, v in row.items()}
            for b, row in out.to_dict(orient="index").items()}


In [ ]:
%%writefile aml/prepare.py
"""Steps 2-3: load, check, attach typologies, partition, persist.

Usage:  python -m aml.prepare --variant HI-Small
        python -m aml.prepare            (both variants, processed independently)
"""
import argparse
import json
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow

from . import config, load, patterns, split


def prepare(variant: str, raw_dir: Path = config.DATA_RAW,
            interim_dir: Path = config.DATA_INTERIM,
            results_dir: Path = config.RESULTS) -> dict:
    t0 = time.perf_counter()
    raw_dir, interim_dir, results_dir = map(Path, (raw_dir, interim_dir, results_dir))

    df = load.read_transactions(raw_dir / f"{variant}_Trans.csv")
    pat = patterns.parse_patterns(raw_dir / f"{variant}_Patterns.txt")
    df, pattern_report = patterns.attach_typology(df, pat)
    df = load.finalise_types(df)

    integrity = load.integrity_checks(df, variant)
    bounds = split.fix_boundaries(df["Timestamp"], results_dir / f"split_{variant}.json")
    df["block"] = split.assign_block(df["Timestamp"], bounds)

    interim_dir.mkdir(parents=True, exist_ok=True)
    df.to_parquet(interim_dir / f"{variant}.parquet", index=False)

    report = {
        "variant": variant,
        "integrity": integrity,
        "patterns": pattern_report,
        "split_boundaries": bounds,
        "blocks": split.block_summary(df),
        "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                        "numpy": np.__version__, "pyarrow": pyarrow.__version__},
        "runtime_seconds": round(time.perf_counter() - t0, 1),
    }
    results_dir.mkdir(parents=True, exist_ok=True)
    (results_dir / f"prepare_{variant}.json").write_text(
        json.dumps(report, indent=2, default=str))
    return report


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant", choices=config.VARIANTS, action="append")
    variants = ap.parse_args().variant or list(config.VARIANTS)
    for v in variants:
        r = prepare(v)
        failed = [k for k, ok in r["integrity"]["checks"].items() if not ok]
        print(f"{v}: {r['integrity']['n_records']:,} records, "
              f"{r['integrity']['n_laundering']:,} laundering, "
              f"span {r['integrity']['span_days']} days, {r['runtime_seconds']} s")
        print(f"  failed checks: {failed or 'none'}  -> results/prepare_{v}.json")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile aml/eda.py
"""Temporal EDA that must precede the freezing of Sections 4.6 (Sets B and C).

It answers three questions on which the feature specification depends:
  1. How long is the observed period, and is laundering evenly spread over it?
  2. Does a rolling window of a given length ever bind, or is it cumulative in practice?
  3. With snapshots at a given frequency, how many transactions have no snapshot at all?

Usage:  python -m aml.eda --variant HI-Small
"""
import argparse
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from . import config

WINDOWS = ["1D", "7D", "30D"]
SNAPSHOT_FREQS = ["12h", "1D", "7D"]


def window_truncation(ts: pd.Series, start: pd.Timestamp) -> dict:
    """Share of transactions whose look-back window reaches before the first record,
    i.e. for which the 'rolling' aggregate is in fact an expanding one."""
    return {w: round(float((ts - pd.Timedelta(w) < start).mean()), 4) for w in WINDOWS}


def snapshot_coverage(df: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> dict:
    """A transaction at t inherits the latest snapshot s <= t, built from records < s.
    Reports, per block, the share of transactions that precede the first snapshot."""
    out = {}
    for f in SNAPSHOT_FREQS:
        first = start.floor("D") + pd.Timedelta(f)
        n_snap = len(pd.date_range(first, end, freq=f))
        no_snap = df["Timestamp"] < first
        out[f] = {"n_snapshots": n_snap,
                  "share_without_snapshot": {
                      b: round(float(no_snap[df["block"] == b].mean()), 4)
                      for b in df["block"].cat.categories}}
    return out


def history_availability(df: pd.DataFrame) -> dict:
    """Share of transactions, per block, whose account was active strictly earlier."""
    parts = []
    for role in ["src_account", "dst_account"]:
        first = df.groupby(role, observed=True)["Timestamp"].min()
        first.index = first.index.astype(str)   # the two roles have different categories
        parts.append(first)
    first_seen = pd.concat(parts).groupby(level=0).min()
    out = {}
    for role in ["src_account", "dst_account"]:
        seen = df[role].astype(str).map(first_seen) < df["Timestamp"]
        out[role] = {b: round(float(seen[df["block"] == b].mean()), 4)
                     for b in df["block"].cat.categories}
    return out


def plot_timeline(df: pd.DataFrame, bounds: dict, path: Path, variant: str) -> None:
    hourly = (df["Is Laundering"].astype("int64").set_axis(df["Timestamp"])
              .resample("1h").agg(["size", "sum"]))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(hourly.index, hourly["size"], lw=0.8, label="transactions per hour")
    ax.set_ylabel("transactions per hour")
    ax2 = ax.twinx()
    ax2.plot(hourly.index, hourly["sum"], lw=0.8, color="tab:red",
             label="laundering per hour")
    ax2.set_ylabel("laundering transactions per hour")
    for key, name in [("cut_train", "train | val"), ("cut_val", "val | test")]:
        ax.axvline(pd.Timestamp(bounds[key]), color="k", ls="--", lw=0.8)
        ax.text(pd.Timestamp(bounds[key]), ax.get_ylim()[1], name, ha="center",
                va="bottom", fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.set_title(f"{variant}: activity over the observed period", pad=16)
    fig.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def run(variant: str, interim_dir: Path = config.DATA_INTERIM,
        results_dir: Path = config.RESULTS) -> dict:
    interim_dir, results_dir = Path(interim_dir), Path(results_dir)
    cols = ["Timestamp", "src_account", "dst_account", "Is Laundering", "block"]
    df = pd.read_parquet(interim_dir / f"{variant}.parquet", columns=cols)
    bounds = json.loads((results_dir / f"split_{variant}.json").read_text())
    start, end = df["Timestamp"].min(), df["Timestamp"].max()

    y = df["Is Laundering"].astype("int64")
    daily = y.groupby(df["Timestamp"].dt.floor("D")).agg(
        n_records="size", n_laundering="sum")
    daily["laundering_per_1000"] = (1000 * daily["n_laundering"] / daily["n_records"]).round(3)
    daily.to_csv(results_dir / f"eda_daily_{variant}.csv")

    report = {
        "variant": variant,
        "first": str(start), "last": str(end),
        "span_days": round((end - start) / pd.Timedelta(days=1), 3),
        "window_truncated_share": window_truncation(df["Timestamp"], start),
        "snapshot_coverage": snapshot_coverage(df, start, end),
        "account_seen_before_share": history_availability(df),
    }
    (results_dir / f"eda_{variant}.json").write_text(json.dumps(report, indent=2))
    plot_timeline(df, bounds, results_dir / f"eda_timeline_{variant}.png", variant)
    return report


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant", choices=config.VARIANTS, action="append")
    for v in ap.parse_args().variant or list(config.VARIANTS):
        r = run(v)
        print(json.dumps(r, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile aml/features.py
"""Feature construction (Section 4.6).

Only the transaction-level attributes of Set A are defined so far. Sets B and C are
added once their window and snapshot specification is fixed.
"""
import numpy as np
import pandas as pd


def _same_value(a: pd.Series, b: pd.Series) -> np.ndarray:
    """Element-wise equality of two categoricals with different category sets,
    compared on shared codes rather than by materialising millions of strings."""
    cats = a.cat.categories.union(b.cat.categories)
    return (a.cat.set_categories(cats).cat.codes.to_numpy()
            == b.cat.set_categories(cats).cat.codes.to_numpy())


def base_attributes(df: pd.DataFrame) -> pd.DataFrame:
    """Set A: attributes available from the transaction record in isolation (Table 4.2).

    Returned untransformed; logarithms, encoding and scaling belong to preprocessing,
    whose parameters are estimated on the training block only.
    """
    ts = df["Timestamp"]
    return pd.DataFrame({
        "amount_paid": df["Amount Paid"].astype("float64"),
        "amount_received": df["Amount Received"].astype("float64"),
        "payment_currency": df["Payment Currency"],
        "receiving_currency": df["Receiving Currency"],
        "currency_mismatch": (~_same_value(df["Payment Currency"],
                                           df["Receiving Currency"])).astype("int8"),
        "payment_format": df["Payment Format"],
        "hour": ts.dt.hour.astype("int8"),
        "day_of_week": ts.dt.dayofweek.astype("int8"),
        "cross_bank": (~_same_value(df["From Bank"], df["To Bank"])).astype("int8"),
    }, index=df.index)


In [ ]:
%%writefile tests/__init__.py



In [ ]:
%%writefile tests/synthetic.py
"""Small synthetic files in the AMLworld layout, for tests only (not for results)."""
from pathlib import Path

import numpy as np
import pandas as pd

HEADER = ("Timestamp,From Bank,Account,To Bank,Account,Amount Received,"
          "Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering")
TYPES = ["FAN-OUT", "FAN-IN", "CYCLE:  Max 4 hops", "STACK"]


def write_synthetic(raw_dir: Path, variant: str = "HI-Small", n: int = 5000,
                    seed: int = 0, days: int = 10) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2022-09-01")
    minutes = np.sort(rng.integers(0, days * 24 * 60, n))
    ts = (start + pd.to_timedelta(minutes, unit="m")).strftime("%Y/%m/%d %H:%M")
    accts = np.array([f"80{i:07X}" for i in range(400)])
    banks = rng.integers(1, 60, size=(n, 2))
    amount = rng.lognormal(7, 2, n).round(2)
    df = pd.DataFrame({
        "Timestamp": ts,
        "From Bank": [f"{b:03d}" for b in banks[:, 0]],
        "src": rng.choice(accts, n), "To Bank": [f"{b:03d}" for b in banks[:, 1]],
        "dst": rng.choice(accts, n),
        "Amount Received": amount, "Receiving Currency": "US Dollar",
        "Amount Paid": amount, "Payment Currency": "US Dollar",
        "Payment Format": rng.choice(["ACH", "Cheque", "Credit Card", "Wire"], n),
        "Is Laundering": 0,
    })
    pos = rng.choice(n, size=max(8, n // 500), replace=False)
    df.loc[pos, "Is Laundering"] = 1

    raw_dir = Path(raw_dir)
    raw_dir.mkdir(parents=True, exist_ok=True)
    with open(raw_dir / f"{variant}_Trans.csv", "w") as fh:
        fh.write(HEADER + "\n")
        df.to_csv(fh, header=False, index=False)

    # Patterns file: bank ids padded to four digits, unlike the transaction file.
    lines, chunks = [], np.array_split(np.sort(pos), 4)
    for k, idx in enumerate(chunks):
        t = TYPES[k % len(TYPES)]
        lines.append(f"BEGIN LAUNDERING ATTEMPT - {t}")
        for i in idx:
            r = df.loc[i].copy()
            r["From Bank"] = f"{int(r['From Bank']):04d}"
            r["To Bank"] = f"{int(r['To Bank']):04d}"
            lines.append(",".join(str(v) for v in r.values))
        lines.append(f"END LAUNDERING ATTEMPT - {t.split(':')[0]}")
        lines.append("")
    (raw_dir / f"{variant}_Patterns.txt").write_text("\n".join(lines))
    return df


In [ ]:
%%writefile tests/test_prepare.py
import json

import pandas as pd
import pytest

from aml import config, load, patterns, prepare, split
from tests.synthetic import write_synthetic


@pytest.fixture
def raw(tmp_path):
    src = write_synthetic(tmp_path / "raw")
    return tmp_path, src


def test_direction_preserved(raw):
    tmp, src = raw
    df = load.read_transactions(tmp / "raw" / "HI-Small_Trans.csv")
    assert (df["src_account"].values == src["src"].values).all()
    assert (df["dst_account"].values == src["dst"].values).all()


def test_unexpected_header_is_refused(raw):
    tmp, _ = raw
    p = tmp / "raw" / "HI-Small_Trans.csv"
    text = p.read_text().replace("From Bank,Account,To Bank", "To Bank,Account,From Bank", 1)
    p.write_text(text)
    with pytest.raises(ValueError):
        load.read_transactions(p)


def test_patterns_join_despite_bank_padding(raw):
    tmp, src = raw
    df = load.read_transactions(tmp / "raw" / "HI-Small_Trans.csv")
    pat = patterns.parse_patterns(tmp / "raw" / "HI-Small_Patterns.txt")
    out, rep = patterns.attach_typology(df, pat)
    assert rep["n_pattern_transactions_unmatched"] == 0
    assert rep["n_typology_rows_labelled_licit"] == 0
    assert rep["n_laundering_rows_without_typology"] == 0
    assert rep["unknown_typology_labels"] == []
    assert set(out["typology"].dropna()) == {"fan-out", "fan-in", "cycle", "stack"}
    assert "typology" in config.NON_FEATURE_COLS


def test_split_is_chronological_and_tie_consistent(raw):
    tmp, _ = raw
    df = load.finalise_types(load.read_transactions(tmp / "raw" / "HI-Small_Trans.csv"))
    b = split.fix_boundaries(df["Timestamp"], tmp / "results" / "split.json")
    df["block"] = split.assign_block(df["Timestamp"], b)
    codes = df["block"].cat.codes
    assert codes.is_monotonic_increasing                     # train < val < test in time
    assert df.groupby("Timestamp")["block"].nunique().max() == 1   # no minute is split
    share = df["block"].value_counts(normalize=True)
    assert abs(share["train"] - 0.70) < 0.02 and abs(share["test"] - 0.15) < 0.02


def test_boundaries_cannot_be_revised(raw):
    tmp, _ = raw
    df = load.finalise_types(load.read_transactions(tmp / "raw" / "HI-Small_Trans.csv"))
    path = tmp / "results" / "split.json"
    split.fix_boundaries(df["Timestamp"], path)
    split.fix_boundaries(df["Timestamp"], path)              # identical: accepted
    with pytest.raises(RuntimeError):
        split.fix_boundaries(df["Timestamp"] + pd.Timedelta("1h"), path)


def test_prepare_end_to_end(raw):
    tmp, _ = raw
    rep = prepare.prepare("HI-Small", tmp / "raw", tmp / "interim", tmp / "results")
    assert (tmp / "interim" / "HI-Small.parquet").exists()
    assert rep["integrity"]["checks"]["timestamps_parsed_and_ordered"]
    saved = json.loads((tmp / "results" / "prepare_HI-Small.json").read_text())
    assert set(saved["blocks"]) == {"train", "val", "test"}


def test_join_diagnostics_detect_inconsistencies(raw):
    tmp, src = raw
    p = tmp / "raw" / "HI-Small_Trans.csv"
    lines = p.read_text().splitlines()
    pos = [i for i, line in enumerate(lines[1:], start=1) if line.endswith(",1")]
    lines[pos[0]] = lines[pos[0]][:-1] + "0"      # pattern transaction labelled licit
    del lines[pos[1]]                              # pattern transaction absent from file
    p.write_text("\n".join(lines) + "\n")
    df = load.read_transactions(p)
    pat = patterns.parse_patterns(tmp / "raw" / "HI-Small_Patterns.txt")
    _, rep = patterns.attach_typology(df, pat)
    assert rep["n_typology_rows_labelled_licit"] == 1
    assert rep["n_pattern_transactions_unmatched"] == 1


In [ ]:
%%writefile tests/test_features.py
import pandas as pd

from aml.features import base_attributes


def test_base_attributes():
    df = pd.DataFrame({
        "Timestamp": pd.to_datetime(["2022-09-01 13:05", "2022-09-03 00:00"]),
        "Amount Paid": [10.0, 5.0], "Amount Received": [10.0, 4.5],
        "Payment Currency": pd.Categorical(["US Dollar", "Euro"]),
        "Receiving Currency": pd.Categorical(["US Dollar", "US Dollar"]),
        "Payment Format": pd.Categorical(["ACH", "Wire"]),
        "From Bank": pd.Categorical(["10", "20"]),
        "To Bank": pd.Categorical(["10", "30"]),
    })
    a = base_attributes(df)
    assert a["currency_mismatch"].tolist() == [0, 1]
    assert a["cross_bank"].tolist() == [0, 1]
    assert a["hour"].tolist() == [13, 0]
    assert a["day_of_week"].tolist() == [3, 5]       # Thursday, Saturday
    assert len(a.columns) == 9                       # Table 4.2: about ten


In [ ]:
%%writefile tests/test_acquire.py
import json

import pytest

from aml import acquire


def fake_kaggle(tmp_path, latest=8):
    """Imitates kagglehub.dataset_download: returns a cache path containing the version."""
    calls = []

    def download(handle, path):
        calls.append(handle)
        v = handle.rsplit("/versions/", 1)[1] if "/versions/" in handle else latest
        p = tmp_path / "cache" / "datasets" / "o" / "d" / "versions" / str(v) / path
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(f"content of {path}")
        return str(p)
    return download, calls


def test_first_download_pins_version_and_later_ones_use_it(tmp_path):
    download, calls = fake_kaggle(tmp_path)
    raw, vfile = tmp_path / "raw", tmp_path / "results" / "DATA_VERSION.json"
    v = acquire.fetch_raw(["A.csv"], raw, vfile, handle="o/d", download=download)
    assert v == 8 and (raw / "A.csv").read_text() == "content of A.csv"
    rec = json.loads(vfile.read_text())
    assert (rec["version"], rec["source"]) == (8, "kagglehub")
    acquire.fetch_raw(["A.csv", "B.txt"], raw, vfile, handle="o/d", download=download)
    assert calls == ["o/d", "o/d/versions/8"]        # A.csv not fetched again; B pinned


def test_version_mismatch_is_refused(tmp_path):
    download, _ = fake_kaggle(tmp_path)
    vfile = tmp_path / "DATA_VERSION.json"
    vfile.write_text(json.dumps({"handle": "o/d", "version": 3}))

    def wrong(handle, path):
        return download("o/d/versions/9", path)
    with pytest.raises(ValueError):
        acquire.fetch_raw(["A.csv"], tmp_path / "raw", vfile, handle="o/d", download=wrong)


def test_manual_files_and_zips_need_no_download(tmp_path):
    import zipfile
    raw, vfile = tmp_path / "raw", tmp_path / "DATA_VERSION.json"
    raw.mkdir()
    (raw / "A.csv").write_text("a")
    with zipfile.ZipFile(raw / "B.txt.zip", "w") as zf:      # as delivered by the website
        zf.writestr("B.txt", "b")

    def no_download(handle, path):
        raise AssertionError("must not download")
    v = acquire.fetch_raw(["A.csv", "B.txt"], raw, vfile, handle="o/d", download=no_download)
    assert v is None and (raw / "B.txt").read_text() == "b"
    assert json.loads(vfile.read_text())["source"] == "placed in data/raw by hand"


def test_failed_download_explains_the_manual_route(tmp_path):
    def unauthorised(handle, path):
        raise PermissionError("401")
    with pytest.raises(RuntimeError, match="upload them"):
        acquire.fetch_raw(["A.csv"], tmp_path / "raw", tmp_path / "v.json",
                          handle="o/d", download=unauthorised)


In [ ]:
# Use the modules just written: discard any copy imported earlier in this session.
# (Replaces IPython's autoreload extension, which fails on Colab's Python 3.12 runtime
#  with "No module named 'imp'".)
import importlib
for name in [n for n in list(sys.modules) if n.split(".")[0] in ("aml", "tests")]:
    del sys.modules[name]
importlib.invalidate_caches()

In [ ]:
# Run the test suite (synthetic fixtures only) before touching the real data
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"],
                     capture_output=True, text=True)
print(res.stdout[-2000:], res.stderr[-2000:])
assert res.returncode == 0, "tests failed"

## 1 Data

### 1.1 Acquisition

Files already in `data/raw/` on Drive (or zip archives containing them) are used as they
are; this is the route for a manual download. Anything missing is downloaded, unmodified,
with `kagglehub` and copied into `data/raw/` (kagglehub's own cache lives on the Colab machine and is wiped between
sessions). The dataset version resolved on the first download is recorded in
`results/DATA_VERSION.json` and requested explicitly on every later download, and the
checksums are written once and verified on every run. Parsing does not use kagglehub's
pandas adapter: its default reader would rename the repeated `Account` column silently
(Section 4.4), and it cannot read the patterns file.

In [ ]:
# Data: download the transaction and pattern files, then record or verify checksums
from aml import acquire, config

RAW = PROJECT / "data" / "raw"
FILES = [f"{v}_{kind}" for v in VARIANTS for kind in ("Trans.csv", "Patterns.txt")]

if DRY_RUN:
    from tests.synthetic import write_synthetic
    for i, v in enumerate(VARIANTS):
        if not (RAW / f"{v}_Trans.csv").exists():
            write_synthetic(RAW, variant=v, n=200_000, seed=i)
else:
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
    try:
        version = acquire.fetch_raw(FILES, RAW, PROJECT / "results" / "DATA_VERSION.json")
    except RuntimeError as e:
        print(e)
        raise SystemExit("Raw files missing: follow the instructions above.")
    print(f"{config.DATASET_HANDLE}, version: {version or 'files placed by hand'}")

absent = [f for f in FILES if not (RAW / f).exists()]
assert not absent, f"missing files: {absent}"

def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while block := fh.read(chunk):
            h.update(block)
    return h.hexdigest()

ck = PROJECT / "results" / "CHECKSUMS.txt"
current = {f: sha256(RAW / f) for f in FILES}
if ck.exists():
    stored = dict(line.split("  ")[::-1] for line in ck.read_text().splitlines() if line)
    changed = [f for f in FILES if f in stored and stored[f] != current[f]]
    assert not changed, f"raw files changed since the first run: {changed}"
    new = {f: h for f, h in current.items() if f not in stored}
    if new:
        with open(ck, "a") as fh:
            fh.writelines(f"{h}  {f}\n" for f, h in new.items())
else:
    ck.write_text("".join(f"{h}  {f}\n" for f, h in current.items()))
for f in FILES:
    print(f"{(RAW / f).stat().st_size / 2**20:8.1f} MB  {current[f][:16]}...  {f}")

### 1.2 Loading and preparation

Each variant is read, typed, joined to its typology labels, checked and partitioned by a
separate call, so that no decision taken on one is informed by the other (Section 4.4).
The prepared frame is stored as Parquet, so the parsing cost is paid once.

In [ ]:
# Data: load, type, check, attach typologies and partition both variants
from aml import prepare

reports = {}
for v in VARIANTS:
    try:
        import psutil
        print(f"available RAM before {v}: {psutil.virtual_memory().available / 2**30:.1f} GB")
    except ImportError:
        pass
    reports[v] = prepare.prepare(v)
    i = reports[v]["integrity"]
    print(f"{v}: {i['n_records']:,} records, {i['n_laundering']:,} laundering "
          f"(1 in {i['observed_ratio_1_in']}), {i['span_days']} days, "
          f"{reports[v]['runtime_seconds']} s")
    gc.collect()

### 1.3 A first look at the data

In [ ]:
# Data: load the prepared frame of the variant explored below and display a sample
df = pd.read_parquet(PROJECT / "data" / "interim" / f"{EDA_VARIANT}.parquet")
df.sample(5, random_state=SEED)

In [ ]:
# Data: summary information
df.info(memory_usage="deep")

In [ ]:
# Data: unique values of the categorical fields
for c in ["Payment Format", "Payment Currency", "Receiving Currency"]:
    cats = sorted(df[c].dropna().unique().astype(str))
    print(f"{c} ({len(cats)}): {cats}")
    print("-" * 100)
n_acc = len(df["src_account"].cat.categories.union(df["dst_account"].cat.categories))
n_bank = len(df["From Bank"].cat.categories.union(df["To Bank"].cat.categories))
print(f"distinct accounts: {n_acc:,}   distinct banks: {n_bank:,}")
print(f"self-transfers (same account on both sides): "
      f"{(df['src_account'].astype(str) == df['dst_account'].astype(str)).mean():.2%}")

## 2 Data quality

In the course examples, deviations from the documented encoding are corrected in place.
Here nothing is corrected: the three integrity checks of Section 4.4 are reported, and any
discrepancy is carried into Appendix A, since it would bear on every result that follows.

In [ ]:
# Data quality: integrity checks and the patterns join, per variant
rows = {}
for v, r in reports.items():
    i = r["integrity"]
    rows[v] = {**{k: i[k] for k in ["n_records", "n_laundering", "observed_ratio_1_in",
                                    "dev_records", "dev_laundering", "dev_ratio",
                                    "n_unparsed_timestamps", "n_duplicate_records",
                                    "n_nonpositive_amount_paid", "span_days"]},
               **{f"check: {k}": ok for k, ok in i["checks"].items()}}
display(pd.DataFrame(rows))
display(pd.DataFrame({v: {k: val for k, val in r["patterns"].items()
                          if not isinstance(val, (dict, list))}
                      for v, r in reports.items()}))

## 3 Temporal partition

The course examples split the data at random, stratified by the target, after the
exploratory analysis. Two things differ here. The split is chronological (70/15/15 per cent
of records ordered by time, Section 4.5), because features built from an account's history
would otherwise let the past borrow from the future. And the split comes *before* the
exploratory analysis: every statistic that conditions on the label is computed on the
training block only, so that no validation or test label can influence a decision about
feature construction or screening.

In [ ]:
# Partition: composition of the three blocks and activity over time
from aml import eda
temporal_all = {v: eda.run(v) for v in VARIANTS}   # both variants, for the report in Section 6
temporal = temporal_all[EDA_VARIANT]
display(pd.DataFrame(reports[EDA_VARIANT]["blocks"]).T)
display(Image(filename=str(PROJECT / "results" / f"eda_timeline_{EDA_VARIANT}.png")))

## 4 Exploratory data analysis (training block)

In [ ]:
# EDA: training block with the Set A attributes (Table 4.2)
from aml.features import base_attributes

FIG, TAB = PROJECT / "results" / "figures", PROJECT / "results" / "tables"
def save(fig, name):
    fig.savefig(FIG / f"{EDA_VARIANT}_{name}.png", dpi=200, bbox_inches="tight")

train = df[df["block"] == "train"]
A = base_attributes(train)
y = train["Is Laundering"].to_numpy().astype("int64")
for c in ["amount_paid", "amount_received"]:       # log10 for display only
    A[f"log10_{c}"] = np.log10(A[c].where(A[c] > 0))
rng = np.random.default_rng(SEED)
print(f"training block: {len(A):,} transactions, {y.sum():,} laundering")

### 4.1 Target variable

In [ ]:
# EDA: Class balance in the training block
n_total, n_pos = len(y), int(y.sum())
n_neg = n_total - n_pos
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(["Licit (0)", "Laundering (1)"], [n_neg, n_pos], color=sns.color_palette("viridis", 2))
ax.set_yscale("log")
ax.set_ylabel("transactions (log scale)")
for k, n in enumerate([n_neg, n_pos]):
    ax.annotate(f"{n:,}\n({100 * n / n_total:.3f} %)", (k, n), ha="center", va="bottom")
ax.set_title(f"{EDA_VARIANT}: class balance, training block")
save(fig, "class_balance")
plt.show()
print(f"1 laundering transaction in {n_total / n_pos:,.0f}; "
      f"predicting 'licit' everywhere would score an accuracy of {n_neg / n_total:.4%}")

A logarithmic axis is needed for the minority class to be visible at all. The accuracy of
the constant classifier printed above is the reason accuracy is not reported (Section 4.9).

### 4.2 Descriptive statistics

In [ ]:
# EDA: Descriptive statistics of the numeric attributes, overall and by class
num = ["amount_paid", "amount_received", "hour", "day_of_week"]
desc = A[num].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
by_class = (A[["amount_paid", "amount_received"]].groupby(y)
            .describe(percentiles=[0.5]).T.rename(columns={0: "licit", 1: "laundering"}))
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(desc)
    display(by_class)
desc.to_csv(TAB / f"{EDA_VARIANT}_describe.csv")
by_class.to_csv(TAB / f"{EDA_VARIANT}_describe_by_class.csv")

### 4.3 Distributions

In [ ]:
# EDA: Histograms of the numeric attributes (amounts on a log10 scale)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.ravel(), ["log10_amount_paid", "log10_amount_received",
                                  "hour", "day_of_week"]):
    data = A[col].dropna()
    bins = 60 if col.startswith("log10") else np.arange(data.min(), data.max() + 2) - 0.5
    ax.hist(data, bins=bins, edgecolor="black", linewidth=0.3)
    ax.set_title(col)
fig.tight_layout()
save(fig, "histograms")
plt.show()

### 4.4 Categorical attributes conditional on the target

The course examples compare class counts within each category. With one positive per
thousand transactions those bars are invisible, so the laundering rate per category is
shown instead, against the overall rate.

In [ ]:
# EDA: Laundering rate per category
cat_cols = ["payment_format", "payment_currency", "receiving_currency",
            "currency_mismatch", "cross_bank"]
base_rate = 1000 * y.mean()
tables = {}
fig, axes = plt.subplots(len(cat_cols), 1, figsize=(12, 3.6 * len(cat_cols)))
for ax, col in zip(axes, cat_cols):
    g = pd.Series(y, index=A.index).groupby(A[col], observed=True)
    t = pd.DataFrame({"n": g.size(), "n_laundering": g.sum()})
    t["rate_per_1000"] = 1000 * t["n_laundering"] / t["n"]
    t["share_of_laundering"] = t["n_laundering"] / t["n_laundering"].sum()
    t = t.sort_values("rate_per_1000", ascending=False)
    tables[col] = t
    ax.bar(t.index.astype(str), t["rate_per_1000"])
    ax.axhline(base_rate, ls="--", color="k", lw=1, label=f"overall: {base_rate:.2f}")
    ax.set_ylabel("laundering per 1,000")
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
fig.tight_layout()
save(fig, "rate_by_category")
plt.show()
for col, t in tables.items():
    print(col)
    display(t.round(4))
    t.to_csv(TAB / f"{EDA_VARIANT}_rate_by_{col}.csv")

### 4.5 Amounts conditional on the target

In [ ]:
# EDA: Amount distributions by class (densities, as the classes differ by three orders of magnitude)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col in zip(axes, ["log10_amount_paid", "log10_amount_received"]):
    lo, hi = np.nanpercentile(A[col], [0.1, 99.9])
    bins = np.linspace(lo, hi, 60)
    for label, name, color in [(0, "licit", "tab:blue"), (1, "laundering", "tab:red")]:
        ax.hist(A.loc[y == label, col].dropna(), bins=bins, density=True,
                alpha=0.5, color=color, label=name)
    ax.set_xlabel(col)
    ax.set_ylabel("density")
    ax.legend(title="class")
fig.tight_layout()
save(fig, "amount_by_class")
plt.show()

### 4.6 Amounts across currencies

In [ ]:
# EDA: Amount paid by payment currency (each amount is expressed in its own currency)
rank = (pd.Series(rng.random(len(A)), index=A.index)
        .groupby(A["payment_currency"], observed=True).rank(method="first"))
sample = A.loc[rank <= 20_000, ["payment_currency", "log10_amount_paid"]].copy()
sample["payment_currency"] = sample["payment_currency"].astype(str)
order = (sample.groupby("payment_currency")["log10_amount_paid"].median()
         .sort_values().index.tolist())
fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=sample, x="payment_currency", y="log10_amount_paid", order=order,
            showfliers=False, ax=ax)
ax.tick_params(axis="x", rotation=45)
ax.set_title("log10 amount paid by payment currency (up to 20,000 per currency)")
save(fig, "amount_by_currency")
plt.show()

If the medians differ by orders of magnitude, a raw amount is not comparable across
currencies. Tree ensembles can learn the interaction with the currency indicator; the
linear models cannot. Whether Set A should therefore carry the log amount standardised
within its currency (parameters estimated on the training block) is a decision to record
in Section 4.6 before Notebook 02.

### 4.7 Standardised attributes by class

In [ ]:
# EDA: Standardised numeric attributes by class (all laundering, random sample of licit)
cols = ["log10_amount_paid", "log10_amount_received", "hour", "day_of_week"]
neg = np.flatnonzero(y == 0)
idx = np.concatenate([np.flatnonzero(y == 1),
                      rng.choice(neg, size=min(100_000, len(neg)), replace=False)])
Z = (A.iloc[idx][cols] - A[cols].mean()) / A[cols].std()
Z["class"] = np.where(y[idx] == 1, "laundering", "licit")
long = Z.melt(id_vars="class", var_name="feature", value_name="value")
fig, ax = plt.subplots(figsize=(14, 6))
sns.violinplot(data=long, x="feature", y="value", hue="class", split=True,
               inner="quart", cut=0, ax=ax)
ax.set_ylim(-3, 3)
ax.set_title("Standardised attributes by class")
save(fig, "violins_by_class")
plt.show()

### 4.8 Correlations

In [ ]:
# EDA: Correlation matrix of the Set A attributes (payment format one-hot) and the target
M = pd.concat([A[["log10_amount_paid", "log10_amount_received", "currency_mismatch",
                  "cross_bank", "hour", "day_of_week"]],
               pd.get_dummies(A["payment_format"], prefix="fmt", dtype="int8"),
               pd.Series(y, index=A.index, name="is_laundering")], axis=1)
corr = M.corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1, annot=True, fmt=".2f",
            square=True, annot_kws={"size": 8}, ax=ax)
ax.set_title("Correlation matrix, training block")
save(fig, "correlations")
plt.show()
corr.to_csv(TAB / f"{EDA_VARIANT}_correlations.csv")

Two readings of this matrix differ. Between features, it is the input to the redundancy
screen in Notebook 02: pairs above |0.95| carry the same information. With the target, it
is not a measure of relevance: the correlation of any feature with a binary target whose
base rate is 0.1 per cent is bounded close to zero, so small values here do not mean that a
feature is useless.

### 4.9 Laundering typologies

In [ ]:
# EDA: Typologies of the laundering transactions across blocks (descriptive only, never a feature)
pos = df.loc[df["Is Laundering"] == 1, ["typology", "block"]].copy()
pos["typology"] = pos["typology"].cat.add_categories("no pattern").fillna("no pattern")
ct = pd.crosstab(pos["typology"], pos["block"], margins=True, margins_name="total")
display(ct)
ct.to_csv(TAB / f"{EDA_VARIANT}_typology_by_block.csv")
fig, ax = plt.subplots(figsize=(10, 4))
ct.drop(index="total")["total"].sort_values().plot.barh(ax=ax)
ax.set_xlabel("laundering transactions")
save(fig, "typologies")
plt.show()

### 4.10 Temporal structure (input to Sets B and C)

In [ ]:
# EDA: Look-back windows, snapshot coverage and account history
print(f"observed period: {temporal['first']} to {temporal['last']} ({temporal['span_days']} days)")
print("\nShare of transactions whose look-back window reaches before the first record:")
display(pd.Series(temporal["window_truncated_share"], name="share").to_frame())
print("Share of transactions preceding the first graph snapshot, by block:")
display(pd.DataFrame({f: v["share_without_snapshot"] for f, v in
                      temporal["snapshot_coverage"].items()}).T
        .assign(n_snapshots=[v["n_snapshots"] for v in temporal["snapshot_coverage"].values()]))
print("Share of transactions whose account was active strictly earlier, by block:")
display(pd.DataFrame(temporal["account_seen_before_share"]).T)
display(pd.read_csv(PROJECT / "results" / f"eda_daily_{EDA_VARIANT}.csv"))

## 5 Findings to record

Answer each point in the text of Section 5.1 and, where it changes the design, in Section 4.6
before Notebook 02 is run:

1. Do the observed counts match Table 4.1? If not, correct the table and note the
   inconsistency of the source.
2. How long is the period, is laundering spread evenly over it, and do the base rates of the
   three blocks differ?
3. Which payment formats and currencies carry rates far above the overall rate?
4. Are amounts on comparable scales across currencies (4.6)?
5. Which feature pairs exceed |0.95| in the correlation matrix (4.8)?
6. In a period of about ten days each weekday occurs once or twice, so `day_of_week` is close
   to an identifier of the calendar date, and the weekdays of the validation and test days
   were seen at most once in training. Keep, or drop from Set A?
7. Which window length and snapshot frequency do the figures in 4.10 support for Sets B and C?

## 6 Output to send back

In [ ]:
# Output: reports for both variants and the list of saved figures and tables
for v in VARIANTS:
    for name in (f"prepare_{v}.json", f"eda_{v}.json"):
        print(f"----- {name}")
        print((PROJECT / "results" / name).read_text())
print("figures:", sorted(p.name for p in FIG.glob("*.png")))
print("tables: ", sorted(p.name for p in TAB.glob("*.csv")))